In [3]:
%reload_ext autoreload
%autoreload 2
import py4DSTEM
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib
from matplotlib import ticker
import os
%matplotlib qt
plt.ion()


In [4]:

trained_model_indicator_index = "sing_02"

model_path = os.getcwd() + "/"
data_path = model_path + "../"
panel_b_path = model_path + "panel_b/"



In [9]:
def rotation_wrt_zAxis(angle_in_rad):
    return np.array(
                    [
                        [np.cos(angle_in_rad), np.sin(angle_in_rad), 0],
                        [-np.sin(angle_in_rad), np.cos(angle_in_rad), 0],
                        [0, 0, 1],
                    ]
                    )


def rotation_wrt_xAxis(angle_in_rad):
    return np.array(
                    [
                        [1, 0, 0],
                        [0, np.cos(angle_in_rad), np.sin(angle_in_rad)],
                        [0, -np.sin(angle_in_rad), np.cos(angle_in_rad)],
                    ]
                    )


def rotMatrix_from_eulerAngles_ZXZ(
                                    angle_z1,
                                    angle_x1, 
                                    angle_z2,
                                    ):
    
    rotationMatrix =    rotation_wrt_zAxis(angle_z1) @ \
                        rotation_wrt_xAxis(angle_x1) @ \
                        rotation_wrt_zAxis(angle_z2)
        
    return rotationMatrix


def calculate_rotation_matrix_for_zone_axis(zone_axis):
    elev = np.arctan2(
        np.hypot(zone_axis[0], zone_axis[1]),
        zone_axis[2],
    )
    azim = np.arctan2(zone_axis[0], zone_axis[1])

    new_rotation_matrix = rotation_wrt_zAxis(azim) @ rotation_wrt_xAxis(elev) @ rotation_wrt_zAxis(-azim)

    return new_rotation_matrix

def sample_zone_axes_on_unit_sphere(number_of_zone_axes_to_sample):
    # Generate N uniform samples for u and theta
    u = np.random.uniform(-1, 1, number_of_zone_axes_to_sample)
    theta = np.random.uniform(0, 2 * np.pi, number_of_zone_axes_to_sample)
    
    # Compute the spherical coordinates for each sample
    x = np.sqrt(1 - u**2) * np.cos(theta)
    y = np.sqrt(1 - u**2) * np.sin(theta)
    z = u
    
    # Stack the coordinates into an Nx3 array
    points = np.vstack((x, y, z)).T
    
    return points

def sample_polarAngles_on_unit_circle(number_of_polar_angles_to_sample):
    # Sample N random angles uniformly between 0 and 2*pi
    theta = np.random.uniform(0, 2 * np.pi, number_of_polar_angles_to_sample)
    return theta

def generate_sphere_surface(radius):
    """Generates the mesh coordinates for a unit sphere surface."""
    # Define a grid of spherical coordinates (u: longitude, v: polar angle)
    u = np.linspace(0, 2 * np.pi, 100) # Longitude (0 to 2*pi)
    v = np.linspace(0, np.pi, 100)     # Polar Angle (0 to pi)

    # Convert spherical coordinates to Cartesian coordinates (x, y, z)
    # x = R * cos(u) * sin(v)
    # y = R * sin(u) * sin(v)
    # z = R * cos(v)
    x = radius * np.outer(np.cos(u), np.sin(v))
    y = radius * np.outer(np.sin(u), np.sin(v))
    z = radius * np.outer(np.ones(np.size(u)), np.cos(v))

    return x, y, z

In [6]:
Cu_cif = "Cu_fcc.cif"

pixel_size = 0.0328
# pixel_size = 0.03158073
sigma_compare = 0.02
pixel_numbers = 128

k_max = pixel_size * pixel_numbers / 2.
accelerating_voltage = int(300e3)
crystal = py4DSTEM.process.diffraction.Crystal.from_CIF(data_path + Cu_cif)
crystal.setup_diffraction(accelerating_voltage)
crystal.calculate_structure_factors(
    k_max,
)

# # Create an orientation plan for [0001] WS2
# crystal.orientation_plan(
#     angle_step_zone_axis = 1,
#     angle_step_in_plane = 1,
#     accel_voltage = 300e3,
#     corr_kernel_size= 0.08, # was 0.08 before 0.12 not bad
#     zone_axis_range='auto',
# )

# Create an orientation plan for [0001] WS2
crystal.orientation_plan(
    angle_step_zone_axis = 2,
    angle_step_in_plane = 2,
    accel_voltage = 300e3,
    corr_kernel_size= 0.08, # was 0.08 before 0.12 not bad
    zone_axis_range='auto',
)



crystal.plot_orientation_zones(
    figsize = (12,12),    
)

crystal.orientation_vecs

Automatically detected point group m-3m,
 using arguments: zone_axis_range = 
[[0 1 1]
 [1 1 1]], 
 fiber_axis=None, fiber_angles=None.
self.orientation_refine
 False 



Orientation plan: 100%|█████████████| 406/406 [00:00<00:00, 4765.53 zone axes/s]


array([[0.        , 0.        , 1.        ],
       [0.        , 0.02908472, 0.99957695],
       [0.0250137 , 0.0250137 , 0.99937412],
       ...,
       [0.53953827, 0.59535639, 0.59535639],
       [0.5585894 , 0.58650571, 0.58650571],
       [0.57735027, 0.57735027, 0.57735027]], shape=(406, 3))

In [21]:
crystal.lat_real[0][0]

np.float64(3.6199)

# ZA sampled for generating synthetic training and validation data

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib
%matplotlib qt
plt.ion()

# --- Configuration ---
RADIUS = 1.0              # Radius of the sphere (Unit Sphere)
GRID_LINE_COLOR = '#76289E'
SPHERE_SURFACE_COLOR = '#DA9DFA'
SPHERE_SURFACE_ALPHA = 0.03
ZA_POINT_COLOR = '#9D4CC7'

REPRESENTATIVE_ZA_COLOR = '#F21616'
REPRESENTATIVE_ZA_EDGECOLOR = '#F21616'

GRID_LINE_ALPHA = 0.3
GRID_LINE_WIDTH = 0.6

ZA_011 = np.array([0., 1., 1.])
ZA_011 = ZA_011/ np.linalg.norm(ZA_011)

ZA_111 = np.array([1., 1., 1.])
ZA_111 = ZA_111/ np.linalg.norm(ZA_111)


# def generate_sphere_points(n_points, radius=RADIUS):
#     """
#     Generates n_points uniformly distributed on the surface of a sphere.
#     Uses random spherical coordinates and converts to Cartesian.
#     """
#     # Generate random phi (azimuthal angle, 0 to 2*pi)
#     phi = np.random.uniform(0, 2 * np.pi, n_points)
#     # Generate random theta (polar angle, 0 to pi). Using arccos(1 - 2*rand) 
#     # ensures uniform distribution over the sphere surface (Lambert projection).
#     theta = np.arccos(2 * np.random.rand(n_points) - 1)

#     # Convert spherical to Cartesian coordinates (r=1)
#     x = radius * np.cos(phi) * np.sin(theta)
#     y = radius * np.sin(phi) * np.sin(theta)
#     z = radius * np.cos(theta)

#     return x, y, z

def plot_3d_sphere_scatter_training_and_validation_data(
                                                        RADIUS = 1.0,
                                                        GRID_LINE_COLOR = '#76289E',
                                                        SPHERE_SURFACE_COLOR = '#DA9DFA',
                                                        SPHERE_SURFACE_ALPHA = 0.04,
                                                        ZA_POINT_COLOR = '#9D4CC7',
                                                        REPRESENTATIVE_ZA_COLOR = '#F21616',
                                                        REPRESENTATIVE_ZA_EDGECOLOR = '#F21616',
                                                        GRID_LINE_ALPHA = 0.5,
                                                        GRID_LINE_WIDTH = 0.8,
):
    """Creates and displays the 3D plot."""
    
    # 1. Prepare Data
    X_surf, Y_surf, Z_surf = generate_sphere_surface(radius=RADIUS)
    # X_points, Y_points, Z_points = generate_sphere_points(NUM_SCATTER_POINTS, radius=RADIUS)
    X_points = crystal.orientation_vecs[:,0]
    Y_points = crystal.orientation_vecs[:,1]
    Z_points = crystal.orientation_vecs[:,2]

    # 2. Setup Plot
    fig = plt.figure(figsize=(12, 12))
    ax = fig.add_subplot(111, projection='3d')
    
    # Set title and background appearance
    # ax.set_title(f'3D Scatter Points on a Unit Sphere Surface (N={NUM_SCATTER_POINTS})', fontsize=14)
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('w')
    ax.yaxis.pane.set_edgecolor('w')
    ax.zaxis.pane.set_edgecolor('w')
    ax.grid(False)

    # 3. Plot the Sphere Surface (with transparency)
    ax.plot_surface(
        X_surf, Y_surf, Z_surf, 
        color=SPHERE_SURFACE_COLOR, 
        alpha=SPHERE_SURFACE_ALPHA,  # Transparency level
        rstride=5, 
        cstride=5, 
        linewidth=0, 
        antialiased=True
    )

    # 4. Plot Spherical Grid Lines

    # 4a. Meridians (Lines of constant longitude, varying polar angle 'v')
    num_meridians = 8
    # Longitude 'u' fixed at various angles (0 to 2*pi)
    meridians_u = np.linspace(0, 2 * np.pi, num_meridians, endpoint=False)
    v_line = np.linspace(0, np.pi, 50) # Polar angle 'v' varies from pole to pole (0 to pi)

    for u in meridians_u:
        x_line = RADIUS * np.cos(u) * np.sin(v_line)
        y_line = RADIUS * np.sin(u) * np.sin(v_line)
        z_line = RADIUS * np.cos(v_line)
        ax.plot(x_line, y_line, z_line, 
                color=GRID_LINE_COLOR, 
                alpha=GRID_LINE_ALPHA, 
                linewidth=GRID_LINE_WIDTH, 
                linestyle='--') # Dashed lines for meridians

    # 4b. Parallels (Lines of constant polar angle 'v', varying longitude 'u')
    num_parallels = 6
    # Polar angle 'v' fixed at various angles. Exclude the poles (v=0 and v=pi)
    parallels_v = np.linspace(0, np.pi, num_parallels + 2)[1:-1]
    u_line = np.linspace(0, 2 * np.pi, 100) # Longitude 'u' varies full circle (0 to 2pi)

    for v in parallels_v:
        # x = R * cos(u) * sin(v_fixed)
        x_line = RADIUS * np.cos(u_line) * np.sin(v)
        # y = R * sin(u) * sin(v_fixed)
        y_line = RADIUS * np.sin(u_line) * np.sin(v)
        # z = R * cos(v_fixed) (z is constant for a given parallel)
        z_line = RADIUS * np.ones_like(u_line) * np.cos(v) 
        ax.plot(x_line, y_line, z_line, 
                color=GRID_LINE_COLOR, 
                alpha=GRID_LINE_ALPHA, 
                linewidth=GRID_LINE_WIDTH, 
                linestyle=':') # Dotted lines for parallels

    # 5. Plot the Scatter Points (on top of the grid)

    ax.scatter(
        X_points, Y_points, Z_points, 
        c=ZA_POINT_COLOR, 
        marker='o', 
        s=7,       # Size of the points
        label=f'sampled zone axes',
        depthshade=False
    )

    ax.scatter(
        0,0,1,
        edgecolors='none', 
        facecolors=REPRESENTATIVE_ZA_COLOR,
        marker='o', 
        alpha = 0.25,
        s=150,       # Size of the points
        label=f'zone axis [001]',
        depthshade=False
    )

    ax.scatter(
        0,0,1,
        edgecolors=REPRESENTATIVE_ZA_EDGECOLOR, facecolors='none',
        linewidths=1.5,
        marker='o', 
        s=180,       # Size of the points
        label=f'zone axis [001]',
        depthshade=False
    )

    

    ax.scatter(
        ZA_011[0],ZA_011[1],ZA_011[2],
        edgecolors='none', 
        facecolors=REPRESENTATIVE_ZA_COLOR,
        marker='o', 
        alpha = 0.25,
        s=150,       # Size of the points
        label=f'zone axis [011]',
        depthshade=False
    )

    ax.scatter(
        ZA_011[0],ZA_011[1],ZA_011[2],
        edgecolors=REPRESENTATIVE_ZA_EDGECOLOR, facecolors='none',
        linewidths=1.5,
        marker='o', 
        s=180,       # Size of the points
        label=f'zone axis [011]',
        depthshade=False
    )

    ax.scatter(
        ZA_111[0],ZA_111[1],ZA_111[2],
        edgecolors='none', 
        facecolors=REPRESENTATIVE_ZA_COLOR,
        marker='o', 
        alpha = 0.25,
        s=150,       # Size of the points
        label=f'zone axis [111]',
        depthshade=False
    )

    ax.scatter(
        ZA_111[0],ZA_111[1],ZA_111[2],
        edgecolors=REPRESENTATIVE_ZA_EDGECOLOR, facecolors='none',
        linewidths=1.5,
        marker='o', 
        s=180,       # Size of the points
        label=f'zone axis [111]',
        depthshade=False
    )
    
    # 6. Configure Axes
    # Set equal aspect ratio for a true spherical representation
    ax.view_init(elev=50, azim=62, roll=-5)
    r_max = RADIUS * 1.01 
    ax.set_xlim([-r_max, r_max])
    ax.set_ylim([-r_max, r_max])
    ax.set_zlim([-r_max, r_max])

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])


    ax.set_axis_off()

    
    # ax.set_xlabel('X Axis')
    # ax.set_ylabel('Y Axis')
    # ax.set_zlabel('Z Axis')
    
    # 7. Show Plot
    # plt.legend()
    plt.savefig('ZA_for_training_and_validation_plot.pdf', dpi=600, bbox_inches='tight')
    plt.savefig('ZA_for_training_and_validation_plot.png', dpi=600, bbox_inches='tight')

    plt.show()


plot_3d_sphere_scatter_training_and_validation_data()

# orientations sampled for generating synthetic test set

In [13]:
np.random.seed(777)  # Set the seed
number_of_orientations_for_test_set = int(10240)



def plot_3d_sphere_scatter_test_data(
                                    panel_b_path,
                                    randomly_sampled_zone_axes,
                                    RADIUS = 1.0,
                                    GRID_LINE_COLOR = '#76289E',
                                    SPHERE_SURFACE_COLOR = '#DA9DFA',
                                    SPHERE_SURFACE_ALPHA = 0.02,
                                    ZA_POINT_COLOR = '#9D4CC7',
                                    REPRESENTATIVE_ZA_COLOR = '#F21616',
                                    REPRESENTATIVE_ZA_EDGECOLOR = '#F21616',
                                    GRID_LINE_ALPHA = 0.95,
                                    GRID_LINE_WIDTH = 1.2,
                                    
):
    """Creates and displays the 3D plot."""
    
    # 1. Prepare Data
    X_surf, Y_surf, Z_surf = generate_sphere_surface(radius=RADIUS)
    X_points = randomly_sampled_zone_axes[:,0]
    Y_points = randomly_sampled_zone_axes[:,1]
    Z_points = randomly_sampled_zone_axes[:,2]

    # 2. Setup Plot
    fig = plt.figure(figsize=(12, 12))
    ax = fig.add_subplot(111, projection='3d')
    
    # Set title and background appearance
    # ax.set_title(f'3D Scatter Points on a Unit Sphere Surface (N={NUM_SCATTER_POINTS})', fontsize=14)
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('w')
    ax.yaxis.pane.set_edgecolor('w')
    ax.zaxis.pane.set_edgecolor('w')
    ax.grid(False)

    # 3. Plot the Sphere Surface (with transparency)
    ax.plot_surface(
        X_surf, Y_surf, Z_surf, 
        color=SPHERE_SURFACE_COLOR, 
        alpha=SPHERE_SURFACE_ALPHA,  # Transparency level
        rstride=5, 
        cstride=5, 
        linewidth=0, 
        antialiased=True
    )

    # 4. Plot Spherical Grid Lines

    # 4a. Meridians (Lines of constant longitude, varying polar angle 'v')
    num_meridians = 8
    # Longitude 'u' fixed at various angles (0 to 2*pi)
    meridians_u = np.linspace(0, 2 * np.pi, num_meridians, endpoint=False)
    v_line = np.linspace(0, np.pi, 50) # Polar angle 'v' varies from pole to pole (0 to pi)

    for u in meridians_u:
        x_line = RADIUS * np.cos(u) * np.sin(v_line)
        y_line = RADIUS * np.sin(u) * np.sin(v_line)
        z_line = RADIUS * np.cos(v_line)
        ax.plot(x_line, y_line, z_line, 
                color=GRID_LINE_COLOR, 
                alpha=GRID_LINE_ALPHA, 
                linewidth=GRID_LINE_WIDTH, 
                linestyle='--') # Dashed lines for meridians

    # 4b. Parallels (Lines of constant polar angle 'v', varying longitude 'u')
    num_parallels = 6
    # Polar angle 'v' fixed at various angles. Exclude the poles (v=0 and v=pi)
    parallels_v = np.linspace(0, np.pi, num_parallels + 2)[1:-1]
    u_line = np.linspace(0, 2 * np.pi, 100) # Longitude 'u' varies full circle (0 to 2pi)

    for v in parallels_v:
        # x = R * cos(u) * sin(v_fixed)
        x_line = RADIUS * np.cos(u_line) * np.sin(v)
        # y = R * sin(u) * sin(v_fixed)
        y_line = RADIUS * np.sin(u_line) * np.sin(v)
        # z = R * cos(v_fixed) (z is constant for a given parallel)
        z_line = RADIUS * np.ones_like(u_line) * np.cos(v) 
        ax.plot(x_line, y_line, z_line, 
                color=GRID_LINE_COLOR, 
                alpha=GRID_LINE_ALPHA, 
                linewidth=GRID_LINE_WIDTH, 
                linestyle=':') # Dotted lines for parallels

    # 5. Plot the Scatter Points (on top of the grid)

    ax.scatter(
        X_points, Y_points, Z_points, 
        c=ZA_POINT_COLOR, 
        marker='o', 
        s=0.04,       # Size of the points
        label=f'sampled zone axes',
        alpha = 0.7,
        depthshade=False
    )

    # 6. Configure Axes
    # Set equal aspect ratio for a true spherical representation
    ax.view_init(elev=50, azim=62, roll=-5)
    r_max = RADIUS * 1.01 
    ax.set_xlim([-r_max, r_max])
    ax.set_ylim([-r_max, r_max])
    ax.set_zlim([-r_max, r_max])

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])


    ax.set_axis_off()

    
    # ax.set_xlabel('X Axis')
    # ax.set_ylabel('Y Axis')
    # ax.set_zlabel('Z Axis')
    
    # 7. Show Plot
    # plt.legend()
    plt.savefig(panel_b_path + 'ZA_for_test_plot.pdf', dpi=300, bbox_inches='tight')
    plt.savefig(panel_b_path + 'ZA_for_test_plot.png', dpi=300, bbox_inches='tight')

    plt.show()

def plot_2d_histogram_of_in_plane_angle(
                                    panel_b_path,
                                    in_plane_angles,
                                    n_bins = 90,
                                    POINT_COLOR = '#9D4CC7',
                                    axes_position = [0.12, 0.22, 0.83, 0.67],
):
    in_plane_angles_degree = in_plane_angles * 180. / np.pi
    # fig, axs = plt.subplots(1, 1, figsize=(10, 5), tight_layout=True)
    fig = plt.figure(figsize=(10, 5), tight_layout=True)
    axs = fig.add_axes(axes_position)
    # We can set the number of bins with the *bins* keyword argument.
    axs.set_title("estimated probability density of sampled in-plane angle", fontsize = 18.5, x=0.44, y=1.05)
    axs.hist(in_plane_angles_degree, range = (0.0, 360), bins=n_bins, density = True,  color = POINT_COLOR, edgecolor = POINT_COLOR,
             # edgecolor='black',
            )
    # axs.hist(pyxemCorrScore_considerInten_transformer, range = (0.0, 0.2), bins=n_bins, alpha = 0.3, color = '#ff0000', label='transf prediction')
    axs.set_xlim([-0.000001, 360])
    axs.set_ylim([-0.000001, 0.0062])
    axs.set_yticks([0.0, 0.003, 0.006])
    axs.set_xticks([0, 90, 180, 270, 360])
    # axs.axvline(x = dominant_angle, color = '#f009f0', linewidth = 3)
    axs.ticklabel_format(axis='y', style='sci', scilimits=(-3, -3))
    axs.set_xlabel("in-plane angle (degree)", fontsize=28)
    axs.set_ylabel("probability density", fontsize=28)
    plt.tick_params(axis = 'both', labelsize = 28, length = 11, width = 0.7, color ='k')
    plt.tight_layout()
    plt.savefig(panel_b_path + "in_plane_angle_for_test_plot.pdf", dpi=600, bbox_inches='tight')
    plt.savefig(panel_b_path + "in_plane_angle_for_test_plot.png", dpi=600, bbox_inches='tight')
    plt.show()

def plot_2d_histogram_of_thickness(
                                    panel_b_path,
                                    thicknesses,
                                    n_bins = 100,
                                    POINT_COLOR = '#9D4CC7',
                                    axes_position = [0.12, 0.22, 0.83, 0.67],
):
    # fig, axs = plt.subplots(1, 1, figsize=(10, 5), tight_layout=True)
    fig = plt.figure(figsize=(8, 6), tight_layout=True)
    axs = fig.add_axes(axes_position)
    # We can set the number of bins with the *bins* keyword argument.
    axs.set_title("estimated probability density of sampled thicknesses", fontsize = 18.5, x=0.44, y=1.05)
    axs.hist(thicknesses, range = (crystal.lat_real[0][0] * 2, crystal.lat_real[0][0] * 900), bins=n_bins, density = True,  color = POINT_COLOR, edgecolor = POINT_COLOR,
             # edgecolor='black',
            )
    # axs.hist(pyxemCorrScore_considerInten_transformer, range = (0.0, 0.2), bins=n_bins, alpha = 0.3, color = '#ff0000', label='transf prediction')
    axs.set_xlim([crystal.lat_real[0][0] * 2, crystal.lat_real[0][0] * 900])
    axs.set_ylim([0.0, 0.0006])
    axs.set_yticks([0.0, 0.0003, 0.0006])
    axs.set_xticks([0, 1000, 2000, 3000])
    axs.ticklabel_format(axis='y', style='sci', scilimits=(-4, -4))
    # axs.axvline(x = dominant_angle, color = '#f009f0', linewidth = 3)
    axs.set_xlabel(r"thickness ($\AA$)", fontsize=28)
    axs.set_ylabel(r"probability density", fontsize=28)
    plt.tick_params(axis = 'both', labelsize = 28, length = 11, width = 0.7, color ='k')
    plt.tight_layout()
    plt.savefig(panel_b_path + "thickness_for_test_plot.pdf", dpi=600, bbox_inches='tight')
    plt.savefig(panel_b_path + "thickness_for_test_plot.png", dpi=600, bbox_inches='tight')
    plt.show()

def plot_2d_density_of_zone_axis(
                                panel_b_path,
                                zone_axes,
                                n_bins = 100,
                                axes_position = [0.12, 0.22, 0.83, 0.67],
):
    x, y, z = zone_axes.T
    azimuth = np.arctan2(y, x)
    azimuth = np.mod(azimuth, 2*np.pi)
    azimuth = azimuth * 180/ np.pi
    cos_polar = z

    fig = plt.figure(figsize=(10.5, 3.5), tight_layout=True)
    axs = fig.add_axes(axes_position)

    plt.hist2d(azimuth, cos_polar, bins=(n_bins, n_bins),
           range=[[0, 360], [-1, 1]], density=True, cmap='Purples')
    # We can set the number of bins with the *bins* keyword argument.
    axs.set_title("zone axis distribution", fontsize = 18.5, x=0.44, y=1.05)
    # axs.hist(thicknesses, range = (crystal.lat_real[0][0] * 2, crystal.lat_real[0][0] * 900), bins=n_bins, density = True,  color = POINT_COLOR, edgecolor = POINT_COLOR,
    #          # edgecolor='black',
    #         )
    # axs.hist(pyxemCorrScore_considerInten_transformer, range = (0.0, 0.2), bins=n_bins, alpha = 0.3, color = '#ff0000', label='transf prediction')
    axs.set_xlim([0,360])
    axs.set_ylim([-1, 1])
    axs.set_yticks([-1, 0, 1])
    axs.set_xticks([0, 90, 180, 270, 360])
    # axs.ticklabel_format(axis='y', style='sci', scilimits=(-4, -4))
    # axs.axvline(x = dominant_angle, color = '#f009f0', linewidth = 3)
    axs.set_xlabel(r'Azimuth $\theta$', fontsize=28)
    axs.set_ylabel(r'cos(polar) = $\cos \phi$', fontsize=28)
    # plt.colorbar(label='Density')
    cbar = plt.colorbar(fraction=0.0468, pad=0.02)
    cbar.set_ticks([0.0, 0.001, 0.002, 0.003])
    cbar.ax.tick_params(labelsize=20)
    cbar.set_label('density', fontsize=20)
    
    formatter = ticker.ScalarFormatter(useMathText=True)
    formatter.set_powerlimits((-3, 3))  # show scientific notation outside this range
    cbar.ax.yaxis.set_major_formatter(formatter)
    plt.tick_params(axis = 'both', labelsize = 28, length = 11, width = 0.7, color ='k')
    plt.tight_layout()
    plt.savefig(panel_b_path + "zone_axis_density_test_set.pdf", dpi=600, bbox_inches='tight')
    plt.savefig(panel_b_path + "zone_axis_density_test_set.png", dpi=600, bbox_inches='tight')
    plt.show()



test_set_data_path = data_path + "generating_test_set/"
zone_axes_b = np.load(test_set_data_path + "Cu_fcc_randomly_sampled_zone_axes_number80000_randSeed777.npy")
thickness_b = np.load(test_set_data_path + "Cu_fcc_randomly_sampled_thicknesses_number80000_randSeed777.npy")
inPlaneAn_b = np.load(test_set_data_path + "Cu_fcc_randomly_sampled_in_plane_angles_number80000_randSeed777.npy")

indices_of_interest = np.load(test_set_data_path + "indices_py4DSTEM_Cu_fcc_ori_table_normalize_by_maxInt_4e3_sg0.040_merged_number65536_randSeed777.npy")

zone_axes = zone_axes_b[indices_of_interest]
thickness = thickness_b[indices_of_interest]
inPlaneAn = inPlaneAn_b[indices_of_interest]

plot_3d_sphere_scatter_test_data(panel_b_path,zone_axes)
plot_2d_histogram_of_in_plane_angle(panel_b_path,inPlaneAn)
plot_2d_histogram_of_thickness(panel_b_path,thickness)
plot_2d_density_of_zone_axis(panel_b_path,zone_axes)





/tmp/ipykernel_30108/3701871100.py:155: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_30108/3701871100.py:185: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_30108/3701871100.py:231: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [14]:
x, y, z = zone_axes.T
azimuth = np.arctan2(y, x)
azimuth = np.mod(azimuth, 2*np.pi)
cos_polar = z

plt.figure(figsize=(11, 5))
plt.hist2d(azimuth, cos_polar, bins=(100, 100),
           range=[[0, 2*np.pi], [-1, 1]], density=True, cmap='Purples')
plt.tick_params(axis = 'both', labelsize = 28, length = 11, width = 0.7, color ='k')
plt.colorbar(label='Density')
plt.xlabel(r'Azimuth $\theta$', fontsize=28)
plt.ylabel(r'cos(polar) = $\cos \phi$', fontsize=28)
plt.title('zone axis distribution')
plt.tight_layout()
plt.savefig("zone_axis_density_test_set.pdf", dpi=600, bbox_inches='tight')
plt.savefig("zone_axis_density_test_set.png", dpi=600, bbox_inches='tight')
plt.show()

x, y, z = zone_axes.T
azimuth = np.arctan2(y, x)
azimuth = np.mod(azimuth, 2*np.pi)
azimuth = azimuth * 180/ np.pi
cos_polar = z

plt.figure(figsize=(11, 5))
plt.hist2d(azimuth, cos_polar, bins=(100, 100),
           range=[[0, 360], [-1, 1]], density=True, cmap='Purples')
plt.tick_params(axis = 'both', labelsize = 28, length = 11, width = 0.7, color ='k')
plt.colorbar(label='Density')
plt.set_xticks([0, 90, 180, 270, 360])
plt.xlabel(r'Azimuth $\theta$', fontsize=28)
plt.ylabel(r'cos(polar) = $\cos \phi$', fontsize=28)
plt.title('zone axis distribution')
plt.tight_layout()
# plt.savefig("zone_axis_density_test_set.pdf", dpi=600, bbox_inches='tight')
# plt.savefig("zone_axis_density_test_set.png", dpi=600, bbox_inches='tight')
plt.show()

AttributeError: module 'matplotlib.pyplot' has no attribute 'set_xticks'

In [79]:
# file_path = "/home/kwang/Desktop/Storage/project/p03_orientation_mapping/figure_for_paper/figure_02/gen_orientations/"
# zone_axes_b = np.load(file_path + "Cu_fcc_randomly_sampled_zone_axes_number80000_randSeed777.npy")
# thickness_b = np.load(file_path + "Cu_fcc_randomly_sampled_thicknesses_number80000_randSeed777.npy")
# inPlaneAn_b = np.load(file_path + "Cu_fcc_randomly_sampled_in_plane_angles_number80000_randSeed777.npy")

# indices_of_interest = np.load("./indices_py4DSTEM_Cu_fcc_ori_table_normalize_by_maxInt_4e3_sg0.040_merged_number65536_randSeed777.npy")

# zone_axes = zone_axes_b[indices_of_interest]
# thickness = thickness_b[indices_of_interest]
# inPlaneAn = inPlaneAn_b[indices_of_interest]

# plot_3d_sphere_scatter_test_data(zone_axes)
# plot_2d_histogram_of_in_plane_angle(inPlaneAn)
# plot_2d_histogram_of_thickness(thickness)

/tmp/ipykernel_19785/2349936502.py:218: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_19785/2349936502.py:247: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [5]:
zone_axes

'/home/kwang/Desktop/Storage/project/p03_orientation_mapping/figure_for_paper/figure_02/gen_orientations/Cu_fcc_randomly_sampled_zone_axes_number99328_randSeed777.npy'

In [70]:
indices_of_interest

array([    0,     1,     2, ..., 67067, 67068, 67069])

In [72]:
zone_axes = zone_axes_b[indices_of_interest]
thickness = thickness_b[indices_of_interest]
inPlaneAn = inPlaneAn_b[indices_of_interest]

In [76]:
zone_axes[5610]

array([ 0.10779182,  0.99305635, -0.04711705])

In [74]:
rotMat = np.load(file_path + "rot_original_py4DSTEM_Cu_fcc_ori_table_normalize_by_maxInt_4e3_sg0.040_merged_number65536_randSeed777.npy")

In [78]:
rotMat[5610]

array([[-0.98397737, -0.14201922,  0.10779182],
       [ 0.09979625,  0.06228805,  0.99305635],
       [-0.14774723,  0.98790219, -0.04711705]])